<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - Digital Forensics</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 2: Drive Forensics</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - What is Drive Forensics?</h1>

Drive forensics is the field of cybersecurity involved in analyzing drives to find deleted files, metadata or deleted paritions.

This is a specific subsection of digital forensics mainly involved with deleted or obscured files.

Drives is a catch all for Hard Disk Drives (HDD) and Solid State Drives (SSD). Which you may find while searching online.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - How are drives formatted?</h2>

Most Drives used for computers today are first formatted with some sort of partition format. Since drives are just one long list of bytes, there needs to be some way to find data or files on the drive. The two main ways of partitioning a drive are: the new way Global Partition Table (GPT), and the old way Master Boot Record (MBR).

MBR has beginning of the disk saved for the bootloader of an operating system. There are also some other limitations to number of partitions which is the reason as to why it is not used as often now. MBR is still required for computers using non-EFI BIOS.

GPT is the newer partitioning format that is used in most newer computers with the introduction of EFI. This allows a boot partition anywhere in the disk. GPT does not have a limit to the number of partitions.

Both MBR and GPT are just a way to organize a drive. Paritions are a way to break a drive up into smaller drives. 

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - How are files stored on drives?</h2>

As said before drives are just a long stream of bytes, so there needs to be a way to find your files somewhere in the list. This is where filesystems come into play. Drives or partitions are formatted with some sort of filesystem (NTFS for Windows, APFS for MacOS, and ext4, FAT32, etc for Linux). These are all just a way of organizing where the data for files is stored and how directories are implemented.

To reduce operations on any drive most filesystems when deleting a file just erase the structures saying where the file is stored on the drive, marking it as unused. Since the data is not overwritten immediately, there is a chance a deleted file is not actually deleted or rewritten.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.3 - What are disk images?</h2>

When professionals are doing drive forensics they first take a drive and plug it into a write blocker. This is a physical piece of hardware that stops their computer from writing to the drive to preserve any possible evidence in the drive. They then make a mirror of the drive called a disk image. These are basically files that are just every byte of the drive in sequence written out.

Since this is what most professionals use, this is the standard way to do drive forensics and disk images is what all of the challenges will be using.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.4 - How to mount disk images on linux from terminal:</h2>

If the disk image is just one file system and there are no partitions you can just mount the entire disk image using the command:
```sh
sudo mount -o loop <disk image here> /path/to/mount/location
# if this fails run the command below
sudo modprobe loop
```
This has to be ran as root which is why `sudo` is used.
This will create a loop device for the file, which is basically a fake drive on your system, and mount it.

-----

If there are multiple paritions on a disk images and you want to create a loop device for each parition. You can use the command:
```sh 
sudo losetup -Pf <disk image here>
```
This will create a loop device for each parition such as: /dev/loop0p1, /dev/loop0p2. These can all be listed using `lsblk`.
You can then mount the parition you want using:
```sh?¶
sudo mount /dev/loop*p* /path/to/mount/location
```
(replace the *'s with the correct numbers)

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Useful commands (assuming you're on linux):</h1>

- `strings`: lists all human readable strings in a file
- `fdisk -l`: lists all partitions in a drive or file
- `grep`: search for a string in some output
- `lsblk`: list all drives and loop devices on your system.

##### Run this before starting the challenges they are all compressed:

In [3]:
%%bash
xz -d *.xz

### Challege 1:

Check the disk image `./chal1.dd`. the flag you are looking for starts with `RPISEC{`. 
<details>
    <summary><b>Hint</b></summary>
    What does the <code>strings</code> command do?
</details>

In [ ]:
%%bash
cd ./chal1
strings disko-1.dd | grep RPISEC

### Challenge 2:

Look at the disk image `./chal2.dd`. The correct parition is Linux.
<details>
    <summary><b>Hint</b></summary>
    Are there multiple partitions
</details>

In [ ]:
%%bash
cd ./chal2
device=$(losetup -f)
sudo losetup -P $device disko-2.dd
parition="${device}p1"
strings $partition | grep RPISEC
sudo losetup -d $device

### Challenge 3:

Check `./chal3.dd`. now the flag is not plain text.
<details>
    <summary><b>Hint</b></summary>
    You might have to mount the disk image
</details>

In [ ]:
%%bash
cd ./chal3
sudo mount -o loop disko-3.dd /mnt 
gzip -dc /mnt/log/flag.gz
sudo umount /mnt

### Challenge 4:

Look at `./chal4.dd`. 
<details>
    <summary><b>Hint</b></summary>
    You should really be using SleuthKit for this one. Check deleted files
</details>